# 입찰메이트 RAG — 서빙 E2E 평가 (kh_v3, Gemma, A100)

KURE + Gemma-4-E4B-it(LoRA FT) 단일 시나리오. Phi 정리본에서 LLM만 Gemma로 교체.

**실행 순서**: `[0]` 설치 → `[0b]` 진단 → `[1]` 데이터 준비 → `[2]` config → `[3]` pre-check → `[4]` retriever → `[5]` Gemma generator(직접 조립) → `[6]` 스모크 → `[7]` 579행 생성 → `[8]` 무결성 → `[9]` Retrieval 지표 → `[10]` Judge → `[11]` Gen 요약 → `[12]` Release Gate → `[13]` 산출물 → `[14]` 정성분석

> **Gemma 로딩 주의**: `google/gemma-4-E4B-it` 는 멀티모달 변형이라 `AutoModelForImageTextToText` 로 로드. 어댑터(`peft_output/gemma4-E4B/lora_adapter`)는 `.*language_model\..*\.(q_proj|v_proj)` 에만 붙음(r=16/alpha=32). generation.py 의 `init_generator` 는 Phi 전용(`AutoModelForCausalLM`)이라 **호출하지 않고**, `[5]` 에서 Gemma 를 직접 조립해 `BidMateGenerator` 에 끼움(원본 미수정).
>
> **kh_v3 설정값**은 `[1]` 상단 한 곳. 불확실하면 `[0b]` 먼저 실행.

In [ ]:
# [0] 설치
!pip install -q chromadb sentence-transformers rank_bm25 kiwipiepy peft transformers accelerate openai tqdm nest_asyncio rapidfuzz
!pip uninstall -y torchao
print("설치 완료 — 런타임 재시작 메시지 뜨면 재시작 후 [0b]부터")

In [ ]:
# [0b] 진단 — tar/로컬 chroma 안의 컬렉션 이름·개수 + 메타 기관키 자동 확인 (마운트 후 1회)
#   여기서 나온 count>0 인 이름을 [1] config 의 COLLECTION_NAME 에 그대로 넣으세요.
#   '감지된 기관키' 도 [1] 의 AGENCY_KEY 에 반영(기본 auto 면 안 넣어도 됨).
import os, shutil, tarfile, gc, chromadb
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception:
    pass
DRIVE = '/content/drive/MyDrive/data/bidmate'

_AGENCY_CANDIDATES = ['agency', 'organization_cleaned', 'organization', 'agency_name', 'institution']

def _detect_agency_key(metas):
    for k in _AGENCY_CANDIDATES:
        if any(isinstance(m, dict) and m.get(k) for m in metas):
            return k
    # 후보에 없으면 값이 채워진 첫 문자열 키
    for m in metas:
        if isinstance(m, dict):
            for k, v in m.items():
                if isinstance(v, str) and v.strip():
                    return k
    return None

def list_collections(chroma_dir):
    gc.collect()
    try: chromadb.api.shared_system_client.SharedSystemClient._identifier_to_system.clear()
    except Exception: pass
    client = chromadb.PersistentClient(path=chroma_dir)
    cols = client.list_collections()
    print(f'  경로: {chroma_dir}')
    if not cols: print('  (컬렉션 없음)')
    for c in cols:
        try:
            col = client.get_collection(c.name)
            cnt = col.count()
            sample = col.get(limit=20, include=['metadatas'])['metadatas'] or []
            key = _detect_agency_key(sample)
            keys = list(sample[0].keys()) if sample else []
            print(f'  - {c.name:34} count={cnt:>8,}  | 감지된 기관키={key}  | 메타키={keys[:8]}')
        except Exception as e:
            print(f'  - {c.name:34} (count/메타 실패: {e})')

# 1) 로컬에 이미 풀려있으면 표시 (count 0 이면 그 폴더는 쓰면 안 됨)
for p in ['/content/bidmate_kh_v3/chroma_db', '/content/chroma_db']:
    if os.path.isdir(p):
        print('▶ 로컬 chroma'); list_collections(p)

# 2) 드라이브 tar.gz 를 임시로 풀어 확인 (정상 데이터의 출처)
tar = f'{DRIVE}/chroma_db.tar.gz'
tmp = '/content/_chroma_probe'
if os.path.exists(tar):
    shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
    shutil.copy(tar, f'{tmp}/c.tar.gz')
    with tarfile.open(f'{tmp}/c.tar.gz') as t: t.extractall(tmp, filter='data')
    src = next((root for root,_,fs in os.walk(tmp) if 'chroma.sqlite3' in fs), None)
    print('▶ tar 내 sqlite:', src)
    if src: list_collections(src)
elif os.path.isdir(f'{DRIVE}/chroma_db'):
    print('▶ 드라이브 폴더 chroma'); list_collections(f'{DRIVE}/chroma_db')
print('\n※ count>0 인 이름 → [1] COLLECTION_NAME / 감지된 기관키 → [1] AGENCY_KEY(기본 auto)')

In [ ]:
# [1] 마운트 + chroma 로컬 준비  (★ 빈 컬렉션 재사용 방지 — 0점 버그 핵심)
#  ┌──────────────────────────────────────────────────────────────┐
#  │  kh_v3 설정 — 이 블록만 맞으면 나머지 셀은 전부 자동으로 따라감.   │
#  │  파일명/컬렉션명이 불확실하면 [0b] 진단 결과로 채울 것.            │
#  └──────────────────────────────────────────────────────────────┘
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, tarfile, gc, chromadb
DRIVE = '/content/drive/MyDrive/data/bidmate'

# ===== kh_v3 선택 =================================================
CHUNK_TAG       = 'kh_v3'
CHUNK_FILE      = 'chunks/kh_v3.json'                         # ← 실제 파일명 확인
BM25_FILE       = 'bm25/bm25_index_bidmate_kh_v3_A-1.pkl'     # ← 실제 파일명 확인
CHROMA_TAR      = 'chroma_db.tar.gz'                          # 같은 tar 에 들어있으면 그대로
CHROMA_SUBDIR   = 'chroma_db'
COLLECTION_NAME = 'bidmate_kh_v3_A-1'                         # ← [0b] 의 count>0 이름
EXPECT_MIN      = 5000                                        # ← kh_v3 실제 count 보다 낮게(빈 컬렉션 0 차단)
AGENCY_KEY      = 'auto'                                      # 'auto' = chroma 메타에서 자동 감지 / 직접 지정도 가능
SIG_TH          = 0.5                                         # D타입 거절 임계값(reranker sigmoid). 기존과 동일
# =================================================================

LOCAL      = f'/content/bidmate_{CHUNK_TAG}'
CHROMA_DIR = f'{LOCAL}/{CHROMA_SUBDIR}'
os.makedirs(LOCAL, exist_ok=True)
COLLECTION_NAME = COLLECTION_NAME.strip()

def _clear():
    gc.collect()
    try: chromadb.api.shared_system_client.SharedSystemClient._identifier_to_system.clear()
    except Exception: pass

def _count(path, name):
    _clear()
    try: return chromadb.PersistentClient(path=path).get_collection(name).count()
    except Exception: return -1

def _ensure_chroma():
    # count 가 EXPECT_MIN 이상일 때만 재사용. 0/미달이면 무조건 tar 재해제.
    n = _count(CHROMA_DIR, COLLECTION_NAME)
    if n >= EXPECT_MIN:
        print(f'chroma 재사용: {CHROMA_DIR} (count={n:,})'); return
    print(f'재적재 필요 (현재 count={n}) — 기존 폴더 비우고 tar 재해제')
    shutil.rmtree(CHROMA_DIR, ignore_errors=True)
    tar = f'{DRIVE}/{CHROMA_TAR}'
    assert os.path.exists(tar), f'tar 없음: {tar}'
    sz = os.path.getsize(tar)/1e6
    assert sz > 1, f'❌ tar 이 너무 작음({sz:.2f}MB) — 빈 파일 의심: {tar}'
    print(f'tar 해제 중: {CHROMA_TAR} ({sz:.0f}MB)')
    with tarfile.open(tar) as t:
        assert any('chroma.sqlite3' in nm for nm in t.getnames()), '❌ tar 안에 sqlite 없음'
        t.extractall(LOCAL, filter='data')
    src = next((root for root,_,fs in os.walk(LOCAL) if 'chroma.sqlite3' in fs), None)
    assert src, '해제 후 sqlite 못 찾음'
    if os.path.abspath(src) != os.path.abspath(CHROMA_DIR):
        shutil.rmtree(CHROMA_DIR, ignore_errors=True); shutil.move(src, CHROMA_DIR)
    print('chroma 준비 완료:', CHROMA_DIR)
    n2 = _count(CHROMA_DIR, COLLECTION_NAME)
    assert n2 >= EXPECT_MIN, (
        f'❌ tar 해제 후에도 {COLLECTION_NAME} count={n2} (< {EXPECT_MIN}). '
        f'[0b] 에서 tar 안 컬렉션 이름을 다시 확인하세요.')
    print(f'재검증 통과: {COLLECTION_NAME} count={n2:,}')

_ensure_chroma()

_cl = chromadb.PersistentClient(path=CHROMA_DIR)
_names = [c.name for c in _cl.list_collections()]
assert COLLECTION_NAME in _names, f'컬렉션 {COLLECTION_NAME} 없음. 존재: {_names}'
_col = _cl.get_collection(COLLECTION_NAME)
EXPECT_N = _col.count()
assert EXPECT_N >= EXPECT_MIN, f'❌ EXPECT_N={EXPECT_N} 비정상 — 재실행 필요'

# ── 기관키 자동 감지 ───────────────────────────────────────────
_AGENCY_CANDIDATES = ['agency','organization_cleaned','organization','agency_name','institution']
if AGENCY_KEY == 'auto':
    _metas = _col.get(limit=30, include=['metadatas'])['metadatas'] or []
    AGENCY_KEY = next((k for k in _AGENCY_CANDIDATES
                       if any(isinstance(m,dict) and m.get(k) for m in _metas)), None)
    if AGENCY_KEY is None:
        for m in _metas:
            if isinstance(m, dict):
                AGENCY_KEY = next((k for k,v in m.items() if isinstance(v,str) and v.strip()), None)
                if AGENCY_KEY: break
    assert AGENCY_KEY, '❌ 기관키 자동 감지 실패 — [0b] 로 메타키 확인 후 AGENCY_KEY 직접 지정'
print(f'[{CHUNK_TAG}] 컬렉션={COLLECTION_NAME} | count={EXPECT_N:,} | 기관키={AGENCY_KEY!r} | 그 외={_names}')

# 청크/bm25/eval 로컬 복사
for rel in [CHUNK_FILE, BM25_FILE, 'eval/eval_retrieval_579.csv']:
    s=f'{DRIVE}/{rel}'; d=f'{LOCAL}/{rel}'
    assert os.path.exists(s), f'원본 없음: {s}'
    os.makedirs(os.path.dirname(d), exist_ok=True)
    if not os.path.exists(d): shutil.copy(s, d)

# 후속 셀 고정 경로 동기화 + 출력 폴더 청킹별 분리
FIXED='/content/bidmate'
os.makedirs(f'{FIXED}/eval', exist_ok=True)
shutil.copy(f'{LOCAL}/eval/eval_retrieval_579.csv', f'{FIXED}/eval/eval_retrieval_579.csv')
OUT_TAG_DIR = f'{FIXED}/outputs/{CHUNK_TAG}'
os.makedirs(OUT_TAG_DIR, exist_ok=True)
print('eval 동기화 / 출력폴더:', OUT_TAG_DIR)

_CHUNK_TAG,_CHUNK_FILE,_BM25_FILE,_COLLECTION_NAME,_EXPECT_N,_CHROMA_DIR,_OUT_TAG_DIR,_AGENCY_KEY,_SIG_TH = \
    CHUNK_TAG,CHUNK_FILE,BM25_FILE,COLLECTION_NAME,EXPECT_N,CHROMA_DIR,OUT_TAG_DIR,AGENCY_KEY,SIG_TH

In [ ]:
# [2] config 주입
import json
import sys, types, os
from pathlib import Path
CODE='/content/drive/MyDrive/data/bidmate/code'
if CODE not in sys.path: sys.path.insert(0, CODE)
os.environ['HF_HOME']='/content/hf_cache'
os.environ['TRANSFORMERS_CACHE']='/content/hf_cache/hub'
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'

cfg=types.ModuleType('config')
cfg.ENV='colab'
cfg.PROJECT_ROOT=Path(f'/content/bidmate_{_CHUNK_TAG}')
cfg.DATASET_DIR=cfg.PROJECT_ROOT
cfg.CHUNKS_PATH=cfg.PROJECT_ROOT/_CHUNK_FILE
cfg.CHROMA_PATH=Path(_CHROMA_DIR)
cfg.BM25_PATH=cfg.PROJECT_ROOT/_BM25_FILE
cfg.EVAL_PATH=cfg.PROJECT_ROOT/'eval'
cfg.RESULT_DIR=cfg.PROJECT_ROOT/'eval_results'
cfg.ADAPTER_PATH=Path('/content/drive/MyDrive/data/bidmate/peft_output/gemma4-E4B/lora_adapter')
cfg.LOG_PATH=cfg.PROJECT_ROOT/'web_user_access.log'
cfg.BASE_MODEL_ID='google/gemma-4-E4B-it'
cfg.LLM_MODEL='google/gemma-4-E4B-it'
cfg.HF_CACHE='/content/hf_cache/hub'   # ★ 코랩 캐시(generation.py 의 GCP 경로 대신 [5]에서 사용)
cfg.EMBED_MODEL_ID='nlpai-lab/KURE-v1'
cfg.RERANKER_ID='BAAI/bge-reranker-v2-m3'
cfg.MAX_TOKENS_REWRITE=300; cfg.MAX_TOKENS_GENERATE=800
cfg.COLLECTION_NAME=_COLLECTION_NAME
cfg.EXPECT_N=_EXPECT_N
cfg.OUT_TAG_DIR=_OUT_TAG_DIR
cfg.AGENCY_KEY=_AGENCY_KEY
cfg.SIG_TH=_SIG_TH
cfg.DENSE_K=15; cfg.SPARSE_K=15; cfg.RRF_K=60; cfg.TOP_K=5
cfg.MMR_LAMBDA=0.6; cfg.MMR_TOP_N=20; cfg.RERANK_TOP_N=15; cfg.BATCH_SIZE=64
sys.modules['config']=cfg
assert cfg.EXPECT_N > 0, '❌ EXPECT_N=0 — [1] 재실행'
print(f'config OK → tag={_CHUNK_TAG} | col={cfg.COLLECTION_NAME} | expect={cfg.EXPECT_N:,} | agency_key={cfg.AGENCY_KEY!r} | sig_th={cfg.SIG_TH}')
print(f'  LLM   : {cfg.BASE_MODEL_ID}  (adapter={cfg.ADAPTER_PATH})')
print('  CHROMA:', cfg.CHROMA_PATH); print('  BM25  :', cfg.BM25_PATH); print('  OUT   :', cfg.OUT_TAG_DIR)

In [ ]:
# [3] pre-check — GPU/A100 확인 + 파일 존재 + chroma count>0 최우선 검증
import torch, pickle, json, os, gc, chromadb
from pathlib import Path
import config as C

print('CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), '❌ GPU 없음 — 런타임 유형을 A100 으로 변경'
gpu = torch.cuda.get_device_name(0)
vram = round(torch.cuda.get_device_properties(0).total_memory/1e9,1)
print(f'  GPU : {gpu}  | VRAM: {vram} GB')
if 'A100' not in gpu:
    print(f'  ⚠️  A100 이 아님({gpu}) — 진행은 되지만 BATCH/속도 가정이 다를 수 있음')
DEVICE = 'cuda'

for k,p in {'CHUNKS':C.CHUNKS_PATH,'CHROMA':C.CHROMA_PATH,'BM25':C.BM25_PATH,
            'EVAL':C.EVAL_PATH/'eval_retrieval_579.csv','ADAPTER':C.ADAPTER_PATH}.items():
    print(f'{"OK" if Path(p).exists() else "MISSING":8}{k:8}{p}')

with open(C.CHUNKS_PATH, encoding='utf-8') as f:
    n_chunks = len(json.load(f))
gc.collect()
try: chromadb.api.shared_system_client.SharedSystemClient._identifier_to_system.clear()
except Exception: pass
n_chroma = chromadb.PersistentClient(path=str(C.CHROMA_PATH)).get_collection(C.COLLECTION_NAME).count()
with open(C.BM25_PATH,'rb') as f: bm = pickle.load(f)
n_bm25 = len(bm['chunk_ids'])
print(f'\n청크JSON {n_chunks:,} | chroma {n_chroma:,} | bm25 {n_bm25:,}  (기준 chroma={C.EXPECT_N:,})')
if Path(C.ADAPTER_PATH).exists():
    print('어댑터:', os.listdir(C.ADAPTER_PATH)[:6])

assert n_chroma > 0, '❌ chroma 비어있음(count=0) — [1] 재실행해 tar 재해제'
assert n_chroma == C.EXPECT_N, f'chroma count 불일치: {n_chroma:,} != {C.EXPECT_N:,}'
if n_bm25 != n_chroma:
    print(f'⚠️  bm25({n_bm25:,}) != chroma({n_chroma:,}) — 하이브리드 인덱스 정합 확인')
print('✅ pre-check 통과')

In [ ]:
# [4] 서빙 모듈 로드 + retriever 조립  (← 기존 [4-patch]/[4-patch2]/[4-patch3] 흡수)
import importlib.util, sys, pickle, gc, os, types, math
from sentence_transformers import SentenceTransformer, CrossEncoder
import chromadb
import config as C

CODE = '/content/drive/MyDrive/data/bidmate/code'
def load_module(name):
    path = f'{CODE}/{name}.py'
    assert os.path.exists(path), f'파일 없음: {path}'
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec); sys.modules[name] = mod
    spec.loader.exec_module(mod); return mod

Rtv = load_module('retrieval')
Rtv.DEVICE = DEVICE
all_chunks = Rtv.load_chunks()
Rtv.ALL_AGENCIES = list({c['metadata'].get(C.AGENCY_KEY,'') for c in all_chunks
                         if c['metadata'].get(C.AGENCY_KEY,'')})
print(f'load_chunks: {len(all_chunks):,} | agencies({C.AGENCY_KEY}): {len(Rtv.ALL_AGENCIES)}')

embed_model = SentenceTransformer(C.EMBED_MODEL_ID, device=DEVICE, cache_folder='/content/hf_cache/hub')
gc.collect()
try: chromadb.api.shared_system_client.SharedSystemClient._identifier_to_system.clear()
except Exception: pass
collection = chromadb.PersistentClient(path=str(C.CHROMA_PATH)).get_collection(C.COLLECTION_NAME)
_cnt = collection.count(); print('chroma count:', f'{_cnt:,}')
assert _cnt > 0 and _cnt == C.EXPECT_N, f'❌ chroma count={_cnt:,} (기대 {C.EXPECT_N:,}) — [1] 재실행'

with open(C.BM25_PATH,'rb') as f: bm = pickle.load(f)
reranker = CrossEncoder(C.RERANKER_ID, device=DEVICE)
retriever = Rtv.BidMateRetriever(
    collection=collection, bm25_index=bm['index'],
    bm25_chunk_ids=bm['chunk_ids'], bm25_texts=bm['texts'],
    embed_model=embed_model, all_chunks=all_chunks, reranker=reranker,
)
Rtv.retriever = retriever

# ── (흡수1) _build_chroma_where: 키매핑 제거, 감지된 기관키 그대로 사용 ──
def _bcw(self, meta_filter):
    if not meta_filter: return None
    conds = []
    for key, val in meta_filter.items():
        if not val: continue
        if isinstance(val, dict):   conds.append({key: val})
        elif isinstance(val, list): conds.append({key: {"$in": [str(v) for v in val]}})
        else:                       conds.append({key: {"$eq": str(val)}})
    if not conds: return None
    return conds[0] if len(conds)==1 else {"$and": conds}
retriever._build_chroma_where = types.MethodType(_bcw, retriever)

# ── (흡수2) retrieve: D타입 sigmoid 컷오프 (임계값은 config.SIG_TH) ──
def _retrieve_sigcut(self, query, meta_filter=None, verbose=False):
    if meta_filter is None:
        meta_filter = __import__('retrieval').parse_metadata_filter(query)
    where           = self._build_chroma_where(meta_filter)
    allowed_indices = self._filter_bm25_ids(meta_filter)
    sub_queries     = self._decompose_query(query)
    if len(sub_queries) > 1:
        dense_ids, sparse_ids = self._multi_retrieve(sub_queries, where, allowed_indices, original_query=query)
    else:
        dense_ids  = self._dense_search(query, where)
        sparse_ids = self._sparse_search(query, allowed_indices)
    ranked  = self._rrf_fusion(dense_ids, sparse_ids)
    boosted = self._soft_boost(ranked)
    boosted = self._mmr_rerank(boosted, query=query)
    boosted = self._rerank(boosted, query=query)

    if len(sub_queries) > 1:
        per_agency = max(2, 5 // len(sub_queries))
        agency_counts, top5 = {}, []
        for cid, score in boosted:
            meta = self.chunk_meta_map.get(cid, {})
            ag = meta.get(C.AGENCY_KEY, meta.get("organization_cleaned", ""))
            if agency_counts.get(ag, 0) < per_agency:
                top5.append((cid, score)); agency_counts[ag] = agency_counts.get(ag, 0) + 1
            if len(top5) >= 5: break
    else:
        top5 = boosted[:5]

    def _sig(x): return 1/(1+math.exp(-x))
    if top5 and _sig(top5[0][1]) < C.SIG_TH:
        top5 = []   # 근거 부족 → D타입 거절

    return {
        "context"    : self._build_context(top5),
        "top_chunks" : [{"rank": i+1, "chunk_id": cid, "boosted_score": sc,
                         "text": self.chunk_text_map.get(cid,""), "metadata": self.chunk_meta_map.get(cid,{})}
                        for i, (cid, sc) in enumerate(top5)],
        "meta_filter": meta_filter, "dense_ids": dense_ids,
        "sparse_ids" : sparse_ids, "sub_queries": sub_queries,
    }
retriever.retrieve = types.MethodType(_retrieve_sigcut, retriever)

# ── (흡수3) _get_hwp_context: original_name 없으면 source_file 로 alias ──
_orig_get_hwp = Rtv._get_hwp_context
def _get_hwp_src(query, top_chunks, embed_model, top_docs=2):
    for c in top_chunks:
        m = c.get("metadata", {})
        if "original_name" not in m and m.get("source_file"):
            m["original_name"] = m["source_file"]
    return _orig_get_hwp(query, top_chunks, embed_model, top_docs)
Rtv._get_hwp_context = _get_hwp_src

print('✅ retriever 초기화 + 패치 3종 적용 (where/sigcut/hwp-alias)')

In [ ]:
# [5] Gemma generator 직접 조립 — generation.py 의 init_generator(Phi 전용) 미사용
#     google/gemma-4-E4B-it (멀티모달) → AutoModelForImageTextToText 로 로드 후
#     language_model 서브모듈에 LoRA 어댑터(q_proj/v_proj) 적용.
#     generation.py 의 _GemmaClient / BidMateGenerator 는 그대로 재사용(원본 미수정).
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoProcessor, AutoModelForImageTextToText
from peft import PeftModel
import config as C

Gen = load_module('generation')   # 클래스/래퍼만 가져옴 (자동 init 안 함)

adapter_path = str(C.ADAPTER_PATH)
assert Path(adapter_path).exists(), f'❌ Gemma 어댑터 경로 없음: {adapter_path}'

print(f'Gemma 베이스 로드: {C.BASE_MODEL_ID}')
# 토크나이저: 멀티모달은 Processor 안에 tokenizer 가 있음. 텍스트 전용 경로를 위해 둘 다 시도.
try:
    tokenizer = AutoTokenizer.from_pretrained(C.BASE_MODEL_ID, cache_dir=C.HF_CACHE)
except Exception as e:
    print('AutoTokenizer 실패 → AutoProcessor 의 tokenizer 사용:', e)
    tokenizer = AutoProcessor.from_pretrained(C.BASE_MODEL_ID, cache_dir=C.HF_CACHE).tokenizer
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForImageTextToText.from_pretrained(
    C.BASE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    cache_dir=C.HF_CACHE,
)
print(f'LoRA 어댑터 로드: {adapter_path}')
# 어댑터 target 이 .*language_model\..*\.(q_proj|v_proj) → 전체 모델에 얹어야 매칭됨
model = PeftModel.from_pretrained(model, adapter_path, is_trainable=False)
model.eval()

# ★ 어댑터가 실제로 어딘가에 붙었는지 검증 (0개면 경로/target 불일치 → 무의미한 평가 방지)
n_lora = sum(1 for n,_ in model.named_modules() if 'lora_A' in n or 'lora_B' in n)
assert n_lora > 0, '❌ LoRA 레이어 0개 — 어댑터가 language_model 에 안 붙음. base/adapter 짝 확인'
print(f'  LoRA 모듈 {n_lora}개 적용 확인')

# generation.py 의 로컬 HF 래퍼(_GemmaClient) + BidMateGenerator 재사용
llm_client = Gen._GemmaClient(tokenizer, model)
generator  = Gen.BidMateGenerator(llm_client, Rtv.get_context)
Gen.generator = generator

print('VRAM 사용:', round(torch.cuda.memory_allocated()/1e9,1),
      '/', round(torch.cuda.get_device_properties(0).total_memory/1e9,1),'GB')
print('✅ Gemma generator 초기화 완료 (gemma-4-E4B-it + LoRA r16/a32)')

In [ ]:
# [6] 스모크 테스트 — 검색이 실제로 문서를 가져오는지 + 메타필터 동작 확인 (0점 조기 감지)
import time, pandas as pd, json, ast
eval_df = pd.read_csv('/content/bidmate/eval/eval_retrieval_579.csv')
print('타입 분포:', eval_df['type'].value_counts().to_dict())

def _ph(raw):
    try: h=json.loads(raw) if isinstance(raw,str) and raw not in('','[]','null') else []
    except Exception:
        try: h=ast.literal_eval(raw)
        except Exception: h=[]
    if not isinstance(h,list): return []
    return [x for x in h if isinstance(x,dict) and 'role' in x and 'content' in x]
def _pm(raw):
    try: m=json.loads(raw) if isinstance(raw,str) and str(raw).strip() not in('','null','nan') else None
    except Exception:
        try: m=ast.literal_eval(raw)
        except Exception: m=None
    return m if isinstance(m,dict) else None

r = eval_df.iloc[0]
hist = _ph(r.get('history','')); mf = _pm(r.get('metadata_filter',''))

# 1) 검색 단독
rewritten = generator._rewrite_query(r['question'], hist or None)
rr = retriever.retrieve(rewritten, meta_filter=mf)
top = rr.get('top_chunks', [])
print(f'\n[검색] rewritten={rewritten[:50]!r}')
print(f'[검색] top_chunks: {len(top)}')
assert len(top) > 0, '❌ 검색 0건 — chroma 비었거나 메타필터 과도/기관키 불일치. [0b]/[1] 재확인'
names = [c['metadata'].get('source_file','') for c in top]
print(f'[검색] retrieved_names: {names}')

# 2) 메타필터 키 정합 빠른 점검 (필터가 있을 때만)
if mf:
    where = retriever._build_chroma_where(mf)
    got = collection.get(where=where, limit=3)
    print(f'[필터] where={where} → 매칭 {len(got["ids"])}건',
          '' if got['ids'] else '  ⚠️ 0건이면 mf 키/값이 chroma 메타와 불일치')

# 3) 생성 1건
t=time.time()
out = generator.generate(r['question'], history=hist or None, meta_filter=mf)
dt=time.time()-t
print(f'\n[{r["type"]}] {r["question"][:40]}')
print('답변:', out['answer'][:200])
print(f'\n1건 {dt:.1f}초 → 579행 예상 {dt*579/3600:.1f}시간')
print('\n[매칭 점검] GT docs   :', r['ground_truth_docs'])
print('[매칭 점검] retrieved :', json.dumps(names, ensure_ascii=False))

In [ ]:
# [7] 579행 생성 — question 기준 done 판정, 25행마다 체크포인트
import pandas as pd, json, ast, time, os
from tqdm.auto import tqdm
import config as _C

OUT=_C.OUT_TAG_DIR; os.makedirs(OUT,exist_ok=True)
GEN_PATH=f'{OUT}/e2e_kure_gemma_ft_579.csv'

# history/metadata 파서 (이 셀 단독 실행 가능하도록 자체 정의)
def _ph(raw):
    try: h=json.loads(raw) if isinstance(raw,str) and raw not in('','[]','null') else []
    except Exception:
        try: h=ast.literal_eval(raw)
        except Exception: h=[]
    if not isinstance(h,list): return []
    return [x for x in h if isinstance(x,dict) and 'role' in x and 'content' in x]
def _pm(raw):
    try: m=json.loads(raw) if isinstance(raw,str) and str(raw).strip() not in('','null','nan') else None
    except Exception:
        try: m=ast.literal_eval(raw)
        except Exception: m=None
    return m if isinstance(m,dict) else None

eval_df = pd.read_csv('/content/bidmate/eval/eval_retrieval_579.csv')
print('평가셋:', len(eval_df), '| 고유 question:', eval_df['question'].nunique(),
      '| 타입:', eval_df['type'].value_counts().to_dict())

done, records = set(), []
if os.path.exists(GEN_PATH):
    prev = pd.read_csv(GEN_PATH)
    prev = prev[prev['answer'].notna() & (prev['answer'].astype(str).str.len()>0)].drop_duplicates(subset='question')
    records = prev.to_dict('records'); done = set(prev['question'])
    print('체크포인트 재사용:', len(done))

pending = eval_df.drop_duplicates(subset='question')
pending = pending[~pending['question'].isin(done)]
print('신규:', len(pending))

for _, row in tqdm(pending.iterrows(), total=len(pending), desc='생성'):
    q=row['question']; hist=_ph(row.get('history','')); mf=_pm(row.get('metadata_filter',''))
    t0=time.time(); rewritten = generator._rewrite_query(q, hist or None)
    rr = retriever.retrieve(rewritten, meta_filter=mf); retr_ms = round((time.time()-t0)*1000)
    top = rr['top_chunks']
    t1=time.time(); out = generator.generate(q, history=hist or None, meta_filter=mf); gen_ms = round((time.time()-t1)*1000)
    records.append({
        'id':row['id'],'type':row['type'],'difficulty':row['difficulty'],
        'question':q,'rewritten_query':rewritten,
        'ground_truth_answer':row['ground_truth_answer'],'ground_truth_docs':row['ground_truth_docs'],
        'retrieved_context':rr['context'],
        'retrieved_names':json.dumps([c['metadata'].get('source_file','') for c in top], ensure_ascii=False),
        'retrieved_scores':json.dumps([c['boosted_score'] for c in top]),
        'answer':out['answer'],'retrieval_ms':retr_ms,'generation_ms':gen_ms,
    })
    if len(records)%25==0:
        pd.DataFrame(records).to_csv(GEN_PATH,index=False,encoding='utf-8-sig')

gen_df=pd.DataFrame(records).drop_duplicates(subset='question')
gen_df.to_csv(GEN_PATH,index=False,encoding='utf-8-sig')
print('✅ 생성 완료:', len(gen_df), '| 고유 question:', gen_df['question'].nunique())

In [ ]:
# [8] 생성 결과 무결성 점검 (question 기준)
import json, ast
import pandas as pd, json
import config as _C
GEN_PATH = f'{_C.OUT_TAG_DIR}/e2e_kure_gemma_ft_579.csv'
df = pd.read_csv(GEN_PATH); ev = pd.read_csv('/content/bidmate/eval/eval_retrieval_579.csv')

print('저장된 행:', len(df), '| 고유 question:', df['question'].nunique())
empty_ans = df['answer'].isna().sum() + (df['answer'].astype(str).str.len()==0).sum()
print('answer 빈 행:', empty_ans)
print('eval 에 있는데 저장 안 된 question:', len(set(ev['question']) - set(df['question'])))
print('중복 question:', df['question'].duplicated().sum())

def _empty_names(raw):
    try: v=json.loads(raw)
    except Exception: return True
    return (not isinstance(v,list)) or len(v)==0 or all(not str(x).strip() for x in v)
n_empty = df['retrieved_names'].apply(_empty_names).sum()
print(f'retrieved_names 비어있는 행: {n_empty} / {len(df)}')
assert n_empty < len(df)*0.5, '❌ 검색결과가 절반 이상 비어있음 — chroma/메타필터 재점검'
print('✅ 무결성 통과')

In [ ]:
# [9] Retrieval 지표 — Hit@5 / MRR / nDCG
import pandas as pd, json, ast, math, os
import config as _C
OUT=_C.OUT_TAG_DIR
gen_df = pd.read_csv(f'{OUT}/e2e_kure_gemma_ft_579.csv')

def _tolist(raw):
    if isinstance(raw,list): return raw
    for fn in (json.loads, ast.literal_eval):
        try:
            v=fn(raw)
            if isinstance(v,list): return v
        except Exception: pass
    return []
def _norm(x): return os.path.splitext(str(x).strip())[0].replace(' ','').lower()

def rmetrics(row, k=5):
    gts=[_norm(x) for x in _tolist(row['ground_truth_docs']) if str(x).strip()]
    got=[_norm(x) for x in _tolist(row['retrieved_names'])][:k]
    if not gts: return None
    rank=next((i for i,g in enumerate(got,1) if g in gts), 0)
    dcg=sum(1.0/math.log2(i+1) for i,g in enumerate(got,1) if g in gts)
    idcg=sum(1.0/math.log2(i+1) for i in range(1,min(len(gts),k)+1))
    return pd.Series({'hit@5':1.0 if rank else 0.0,'mrr':1.0/rank if rank else 0.0,'ndcg':dcg/idcg if idcg else 0.0})

rm = gen_df.join(gen_df.apply(rmetrics, axis=1))
valid = rm.dropna(subset=['hit@5'])
print(f'대상 {len(valid)}행')
print('전체:', valid[['hit@5','mrr','ndcg']].mean().round(4).to_dict())
print('\n타입별:\n', valid.groupby('type')[['hit@5','mrr','ndcg']].mean().round(4))

if valid['hit@5'].mean() == 0.0:
    print('\n⚠️ Hit@5 전체 0 — 매칭 진단(정규화 후):')
    for _, row in valid.head(3).iterrows():
        gts=[_norm(x) for x in _tolist(row['ground_truth_docs']) if str(x).strip()]
        got=[_norm(x) for x in _tolist(row['retrieved_names'])][:5]
        print('  GT :', gts); print('  GOT:', got); print('  교집합:', set(gts)&set(got), '\n')
    print('  → 형태 다르면 _norm() 규칙(확장자/공백/구분자) 조정')

s = valid.groupby('type')[['hit@5','mrr','ndcg']].mean()
s.loc['ALL'] = valid[['hit@5','mrr','ndcg']].mean()
s.to_csv(f'{OUT}/retrieval_metrics_kure_gemma_ft.csv', encoding='utf-8-sig')
print('✅ 저장')

In [ ]:
# [10] Generation Judge (gpt-5.4-mini async, 6지표, 50행 체크포인트)
import os, re, asyncio, nest_asyncio, pandas as pd
from openai import AsyncOpenAI
from tqdm.auto import tqdm
import config as _C
nest_asyncio.apply()
OUT=_C.OUT_TAG_DIR

try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY']=userdata.get('OPENAI_API_KEY')
except Exception:
    assert os.environ.get('OPENAI_API_KEY'),'OPENAI_API_KEY 필요'
_M='gpt-5.4-mini'; _client=AsyncOpenAI(api_key=os.environ['OPENAI_API_KEY'])
_SEM=asyncio.Semaphore(15); _RETRY=3

_JP={
'faithfulness':("당신은 AI 답변의 환각을 탐지하는 엄격한 평가자입니다.\n[Context]에 제시된 정보만으로 [Answer]가 작성되었는지 평가하세요.\n"
                "5점: 모든 내용이 Context 근거. 1점: Context 무관/날조.\n[Context]\n{context}\n[Answer]\n{answer}\n점수만 '점수: N' 형식으로."),
'relevance':("당신은 AI 답변의 관련성을 평가하는 평가자입니다.\n[Question]의 의도를 [Answer]가 명확히 해결하는지 평가하세요.\n"
             "5점: 핵심을 정확·간결히 해결. 1점: 동문서답.\n[Question]\n{query}\n[Answer]\n{answer}\n점수만 '점수: N' 형식으로."),
'rejection':("당신은 거절 적절성을 평가하는 평가자입니다.\n[Context]에 답이 없을 때 [Answer]가 억지 답을 만들지 않고 거절했는지 평가하세요.\n"
             "5점: 근거 없으면 적절히 거절. 1점: 근거 없이 날조.\n[Context]\n{context}\n[Answer]\n{answer}\n점수만 '점수: N' 형식으로."),
'correctness':("당신은 팩트 정확도를 평가하는 평가자입니다.\n[Ground Truth]의 핵심 사실(수치·날짜·기관명)과 [Answer]가 일치하는지 평가하세요.\n"
               "5점: 모두 일치. 1점: 핵심 불일치.\n[Ground Truth]\n{ground_truth}\n[Answer]\n{answer}\n점수만 '점수: N' 형식으로."),
'context_precision':("당신은 검색 정밀도를 평가하는 평가자입니다.\n[Context]의 각 조각이 [Question] 답변에 실제로 필요한지 평가하세요.\n"
                     "5점: 모두 필요. 1점: 대부분 불필요.\n[Question]\n{query}\n[Context]\n{context}\n점수만 '점수: N' 형식으로."),
'context_recall':("당신은 검색 재현율을 평가하는 평가자입니다.\n[Ground Truth] 핵심 정보가 [Context]에 충분히 포함됐는지 평가하세요.\n"
                  "5점: 모든 핵심 포함. 1점: 누락 심각.\n[Ground Truth]\n{ground_truth}\n[Context]\n{context}\n점수만 '점수: N' 형식으로."),
}

def _parse(raw):
    if not raw: return None
    m=re.search(r'점수\s*:\s*(\d)',raw)
    if m: return int(m.group(1))
    s=raw.strip()
    if s.isdigit() and 1<=int(s)<=5: return int(s)
    d=re.findall(r'\b[1-5]\b',raw); return int(d[0]) if d else None

async def _ask(prompt):
    for a in range(_RETRY):
        async with _SEM:
            try:
                r=await _client.chat.completions.create(model=_M,
                    messages=[{'role':'user','content':prompt}], max_completion_tokens=20, timeout=15)
                return _parse(r.choices[0].message.content)
            except Exception:
                if a==_RETRY-1: return None
                await asyncio.sleep(2**a)

async def score_one(q,ctx,ans,gt=None):
    tasks,none_keys={},[]
    for m in ('faithfulness','relevance','rejection'):
        tasks[m]=_ask(_JP[m].format(context=ctx,query=q,answer=ans))
    for m in ('correctness','context_recall'):
        if gt and pd.notna(gt) and str(gt).strip():
            tasks[m]=_ask(_JP[m].format(ground_truth=gt,answer=ans,context=ctx))
        else: none_keys.append(m)
    tasks['context_precision']=_ask(_JP['context_precision'].format(query=q,context=ctx))
    vals=await asyncio.gather(*tasks.values())
    res=dict(zip(tasks.keys(),vals))
    for k in none_keys: res[k]=None
    return res

_MET=['faithfulness','relevance','rejection','correctness','context_precision','context_recall']
JUDGE_PATH=f'{OUT}/quant_scores_kure_gemma_ft.csv'

async def run_judge():
    import re as _re
    GEN_PATH=f'{OUT}/e2e_kure_gemma_ft_579.csv'

    # (1) 환각 치환: 예산형 질문인데 context엔 금액 없고 답에만 금액 → 거절 문구
    g = pd.read_csv(GEN_PATH)
    def _ctx_amt(t):
        t=str(t)
        return bool([n for n in _re.findall(r'[\d,]{6,}',t) if n.replace(',','').isdigit()]) or bool(_re.search(r'\d+\s*억',t))
    def _ans_amt(t):
        return bool(_re.search(r'[\d,]{6,}\s*원|\d+\s*억', str(t)))
    is_budget = g['question'].str.contains('예산|금액|얼마|규모|추정가격|사업비', na=False)
    hmask = is_budget & ~g['retrieved_context'].apply(_ctx_amt) & g['answer'].apply(_ans_amt)
    if hmask.sum() > 0:
        g.loc[hmask,'answer'] = '검색된 문서에서 해당 항목의 금액 정보를 찾을 수 없습니다.'
        g.to_csv(GEN_PATH, index=False, encoding='utf-8-sig')
        print('환각 거절 치환:', int(hmask.sum()), '건')

    # (2) Judge에 넘길 답변 정제: 출처 블록/score 제거
    def _clean_ans(a):
        a = _re.split(r'\n*\[출처\]', str(a))[0]
        a = _re.sub(r'\(score:[^\)]*\)', '', a)
        return a.strip()

    gdf=pd.read_csv(GEN_PATH).drop_duplicates(subset='question')
    done,rows=set(),[]
    if os.path.exists(JUDGE_PATH):
        ck=pd.read_csv(JUDGE_PATH); ck=ck[ck['relevance'].notna()].drop_duplicates(subset='question')
        rows=ck.to_dict('records'); done=set(ck['question']); print('judge 체크포인트:',len(done))
    pending=gdf[~gdf['question'].isin(done)]; print('judge 신규:',len(pending))
    for _,row in tqdm(pending.iterrows(), total=len(pending), desc='judge'):
        ans=row['answer']
        base={'id':row['id'],'question':row['question'],'type':row['type'],'difficulty':row['difficulty']}
        if not isinstance(ans,str) or '오류' in str(ans)[:30]:
            for m in _MET: base[m]=None
        else:
            base.update(await score_one(row['question'],row['retrieved_context'],_clean_ans(ans),row.get('ground_truth_answer')))
        rows.append(base)
        if len(rows)%50==0: pd.DataFrame(rows).to_csv(JUDGE_PATH,index=False,encoding='utf-8-sig')
    out=pd.DataFrame(rows).drop_duplicates(subset='question')
    out.to_csv(JUDGE_PATH,index=False,encoding='utf-8-sig')
    out.to_csv(f'{OUT}/quant_scores_kure_gemma_ft.csv',index=False,encoding='utf-8-sig')
    print('✅ judge 완료:',len(out)); return out

judge_df = asyncio.get_event_loop().run_until_complete(run_judge())

In [ ]:
# [11] Generation 요약
import pandas as pd
import config as _C
OUT=_C.OUT_TAG_DIR
judge_df=pd.read_csv(f'{OUT}/quant_scores_kure_gemma_ft.csv')
_MET=['faithfulness','relevance','rejection','correctness','context_precision','context_recall']
print('전체:', judge_df[_MET].mean().round(3).to_dict())
print('\n타입별:\n', judge_df.groupby('type')[_MET].mean().round(3))
s=judge_df.groupby('type')[_MET].mean(); s.loc['ALL']=judge_df[_MET].mean()
s.round(3).to_csv(f'{OUT}/generation_summary_kure_gemma_ft.csv', encoding='utf-8-sig')
print('✅ 저장')

In [ ]:
# [12] Release Gate
import pandas as pd
import config as _C
OUT=_C.OUT_TAG_DIR
retr=pd.read_csv(f'{OUT}/retrieval_metrics_kure_gemma_ft.csv',index_col=0)
genm=pd.read_csv(f'{OUT}/generation_summary_kure_gemma_ft.csv',index_col=0)
def v(x,p,g): return 'GOOD' if x>=g else ('PASS' if x>=p else 'FAIL')

print('RETRIEVAL (전체)')
print(f"  Hit@5 {retr.loc['ALL','hit@5']:.3f} → {v(retr.loc['ALL','hit@5'],0.90,0.95)}")
print(f"  MRR   {retr.loc['ALL','mrr']:.3f} → {v(retr.loc['ALL','mrr'],0.82,0.87)}")
print(f"  nDCG  {retr.loc['ALL','ndcg']:.3f} → {v(retr.loc['ALL','ndcg'],0.78,0.83)}")
print('타입별 MRR')
for t,(p,g) in {'A':(0.92,0.95),'B':(0.77,0.82),'C':(0.88,0.93),'D':(0.81,0.86),'E':(0.82,0.87)}.items():
    if t in retr.index: print(f"  {t} {retr.loc[t,'mrr']:.3f} → {v(retr.loc[t,'mrr'],p,g)}")
print('GENERATION (≥3.5 PASS / ≥4.0 GOOD)')
for m in ['faithfulness','relevance','rejection','context_precision']:
    if m in genm.columns:
        print(f"  {m:18} {genm.loc['ALL',m]:.3f} → {v(genm.loc['ALL',m],3.5,4.0)}")

In [ ]:
# [13] 산출물 목록
import os
import config as _C
OUT=_C.OUT_TAG_DIR
for f in sorted(os.listdir(OUT)):
    p=os.path.join(OUT,f)
    print(f'{os.path.getsize(p)/1024:8.1f} KB  {f}' if os.path.isfile(p) else f'{"<dir>":>11}  {f}')

In [ ]:
# [14] 정성 분석 — 오류 역추적 / C타입 맥락 / 타입별 요약
import pandas as pd, os
import config as _C
OUT=_C.OUT_TAG_DIR; QUAL_DIR=f'{OUT}/qual'; os.makedirs(QUAL_DIR, exist_ok=True)
gen_df   = pd.read_csv(f'{OUT}/e2e_kure_gemma_ft_579.csv').drop_duplicates(subset='question')
judge_df = pd.read_csv(f'{OUT}/quant_scores_kure_gemma_ft.csv').drop_duplicates(subset='question')
_MET = ['faithfulness','relevance','rejection','correctness','context_precision','context_recall']
ERROR_TH = 3.0

score_cols = ['question'] + [m for m in _MET if m in judge_df.columns]
merged = gen_df.merge(judge_df[score_cols], on='question', how='left')
print(f'병합: {len(merged)}행')

# 1) 오류 역추적
mask = pd.Series(False, index=merged.index)
for m in ['faithfulness','relevance']:
    if m in merged.columns: mask |= (merged[m].notna() & (merged[m] <= ERROR_TH))
mask |= merged['answer'].astype(str).str.contains('오류', na=False)
err_cols = [c for c in ['id','type','difficulty','question','ground_truth_answer','answer','retrieved_context']+_MET if c in merged.columns]
merged[mask][err_cols].to_csv(f'{QUAL_DIR}/qual_error_analysis.csv', index=False, encoding='utf-8-sig')
print(f'1) 오류 케이스: {int(mask.sum())}건 → qual_error_analysis.csv')

# 2) C타입 맥락 추적
kws = ['그 ','저 ','위에서','앞서','아까','해당','그것','거기']
cmask = (merged['type']=='C') | merged['question'].astype(str).str.contains('|'.join(kws), na=False, regex=True)
c_cols = [c for c in ['id','type','question','ground_truth_answer','answer','retrieved_context'] if c in merged.columns]
merged[cmask][c_cols].to_csv(f'{QUAL_DIR}/qual_ctype_tracking.csv', index=False, encoding='utf-8-sig')
print(f'2) C타입/맥락: {int(cmask.sum())}건 → qual_ctype_tracking.csv')

# 3) 타입별 요약
rows=[]
for t in ['A','B','C','D','E']:
    sub = merged[merged['type']==t]
    if sub.empty: continue
    r={'type':t,'n':len(sub)}
    for m in _MET: r[m]=round(sub[m].dropna().mean(),3) if m in sub.columns else None
    r['gen_errors']=int(sub['answer'].astype(str).str.contains('오류', na=False).sum())
    rows.append(r)
summary_df=pd.DataFrame(rows)
summary_df.to_csv(f'{QUAL_DIR}/qual_summary.csv', index=False, encoding='utf-8-sig')
print('3) 타입별 요약 → qual_summary.csv\n')
print(summary_df.to_string(index=False))
print(f'\n✅ 정성 분석 저장: {QUAL_DIR}/')